<a href="https://colab.research.google.com/github/vahagngrigoryan2006/flyrank-internship-ml/blob/main/work/notebooks/w04_signal_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vahagngrigoryan2006/flyrank-internship-ml/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import os, sys
import pandas as pd
import numpy as np
import duckdb

REPO_URL = "https://github.com/vahagngrigoryan2006/flyrank-internship-ml"
REPO_DIR = "flyrank-internship-ml"

if not os.path.isdir(REPO_DIR):
    import subprocess
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)

con = duckdb.connect()

from google.colab import userdata
HF_TOKEN = userdata.get("HF_TOKEN")
assert HF_TOKEN, "Set HF_TOKEN as a Colab secret (or env var if running locally) before continuing."

con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN \'{HF_TOKEN}\')")

WAREHOUSE = "hf://datasets/FlyRank/internship-warehouse"
DIM_CONTENT     = f"read_parquet(\'{WAREHOUSE}/dim_content.parquet\')"
FACT_APRIL_MAY  = (
    f"read_parquet([\'{WAREHOUSE}/fact_content_daily_performance/month=2026-03/*.parquet\', "
    f"\'{WAREHOUSE}/fact_content_daily_performance/month=2026-04/*.parquet\', "
    f"\'{WAREHOUSE}/fact_content_daily_performance/month=2026-05/*.parquet\'])"
)

# Same window as w04_baseline_score.ipynb: as-of 2026-05-31, decision moment 2026-05-01.
features = con.sql(f"""
    WITH bounds AS (
        SELECT DATE \'2026-05-31\' AS as_of_date
    ),
    per_item AS (
        SELECT f.client_hash_id, f.content_hash_id,
               MIN(f.report_date) AS first_seen,
               SUM(CASE WHEN f.report_date >= b.as_of_date - INTERVAL 30 DAY
                        THEN f.gsc_impressions ELSE 0 END) AS last_30_impressions,
               SUM(CASE WHEN f.report_date <  b.as_of_date - INTERVAL 30 DAY AND f.report_date >= b.as_of_date - INTERVAL 60 DAY
                        THEN f.gsc_impressions ELSE 0 END) AS prev_30_impressions,
               SUM(CASE WHEN f.report_date <  b.as_of_date - INTERVAL 30 DAY AND f.report_date >= b.as_of_date - INTERVAL 60 DAY
                        THEN f.gsc_clicks ELSE 0 END)      AS prev_30_clicks,
               AVG(CASE WHEN f.report_date <  b.as_of_date - INTERVAL 30 DAY AND f.report_date >= b.as_of_date - INTERVAL 60 DAY
                        THEN f.gsc_avg_position END)       AS prev_30_avg_position
        FROM {FACT_APRIL_MAY} f, bounds b
        GROUP BY 1, 2
    )
    SELECT p.*, d.content_created_date, d.content_updated_date
    FROM per_item p
    JOIN {DIM_CONTENT} d USING (content_hash_id)
    WHERE p.first_seen <= DATE \'2026-05-31\' - INTERVAL 60 DAY   -- guard (a): full prev_30 history
      AND p.prev_30_impressions >= 100                             -- guard (b): activity floor
""").df()

decision_moment = pd.Timestamp("2026-05-01")
features["content_created_date"] = pd.to_datetime(features["content_created_date"])
features["content_updated_date"] = pd.to_datetime(features["content_updated_date"])
features["content_age_days_at_decision"] = (decision_moment - features["content_created_date"]).dt.days
features["days_since_last_update_at_decision"] = (decision_moment - features["content_updated_date"]).dt.days
features = features[features["days_since_last_update_at_decision"] > 0]   # guard (c)

features["impressions_pct_change"] = (
    (features["last_30_impressions"] - features["prev_30_impressions"]) / features["prev_30_impressions"]
)
features["is_declining"] = (features["impressions_pct_change"] < -0.20).astype(int)
features["prev_30_ctr"] = features["prev_30_clicks"] / features["prev_30_impressions"] * 100

df = features.copy()
print(f"Working frame: {len(df):,} content items surviving all three guards.")
print(f"is_declining rate: {df['is_declining'].mean():.3f}")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Working frame: 18,918 content items surviving all three guards.
is_declining rate: 0.543


In [2]:
key_fields = ["prev_30_impressions", "prev_30_clicks", "prev_30_ctr", "prev_30_avg_position",
              "content_age_days_at_decision", "days_since_last_update_at_decision"]

desc = df[key_fields].describe(percentiles=[0.5, 0.9, 0.95, 0.99]).T
desc["max_over_p99"] = (desc["max"] / desc["99%"]).round(1)   # a quick, blunt heavy-tail signal
print("Distributions of key fields:")
desc.round(2)

Distributions of key fields:


,count,mean,std,min,50%,90%,95%,99%,max,max_over_p99
prev_30_impressions,18918.0,1570.56,4899.26,100.00,498.00,2986.30,5828.75,19693.80,267960.00,13.6
prev_30_clicks,18918.0,3.56,15.13,0.00,0.00,7.00,15.00,57.00,1024.00,18.0
prev_30_ctr,18918.0,0.19,0.37,0.00,0.00,0.59,0.85,1.62,10.09,6.2
prev_30_avg_position,18918.0,18.64,15.73,0.14,13.37,40.03,51.63,71.59,89.44,1.2
content_age_days_at_decision,18918.0,217.92,78.30,56.00,219.00,308.00,323.00,381.00,406.00,1.1
days_since_last_update_at_decision,18918.0,59.23,20.80,4.00,65.00,65.00,65.00,67.00,295.00,4.4


In [5]:
zero_ctr_share = (df["prev_30_ctr"] == 0).mean()
print(f"Share of rows with EXACTLY zero prev_30 clicks (prev_30_ctr == 0): {zero_ctr_share:.3f}")
print("This is the number that motivated my change from `ctr < 0.5x tier average` to")
print("`ctr == 0` in w04_baseline_score.ipynb")
print()
print("Heavy-tail read: compare `max` to `99%` in the table above for prev_30_impressions and")
print("prev_30_clicks -- a `max_over_p99` well above 1 means a handful of extreme rows sit far")
print("beyond where the bulk of the data lives (the classic heavy-tail shape for traffic data).")

Share of rows with EXACTLY zero prev_30 clicks (prev_30_ctr == 0): 0.524
This is the number that motivated my change from `ctr < 0.5x tier average` to
`ctr == 0` in w04_baseline_score.ipynb

Heavy-tail read: compare `max` to `99%` in the table above for prev_30_impressions and
prev_30_clicks -- a `max_over_p99` well above 1 means a handful of extreme rows sit far
beyond where the bulk of the data lives (the classic heavy-tail shape for traffic data).


## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.